In [0]:
import requests
import boto3
import time
from concurrent.futures import ThreadPoolExecutor
from botocore.exceptions import ClientError
from boto3.s3.transfer import TransferConfig
from pyspark.sql.types import StructType, StructField, StringType

# 1. Credentials
ACCESS_KEY  = dbutils.secrets.get(scope="aws-creds", key="access_key")
SECRET_KEY  = dbutils.secrets.get(scope="aws-creds", key="secret_key")
BUCKET_NAME = "s3-goodreads-data"
RAW_PARQUET = "s3://s3-goodreads-data/Raw/interactions/parquet"

# 2. DOWNLOAD SETUP
TRANSFER_CONFIG = TransferConfig(
    multipart_threshold=1024 * 25,
    multipart_chunksize=1024 * 25,
    max_concurrency=4,
    use_threads=True
)

s3_client = boto3.client('s3', aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY)
base_url = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/"

files_to_download = {
    "goodreads_books.json.gz":               "Raw/metadata/books.json.gz",
    "goodreads_book_authors.json.gz":        "Raw/metadata/authors.json.gz",
    "goodreads_book_genres_initial.json.gz": "Raw/metadata/genres.json.gz",
    "goodreads_interactions_dedup.json.gz":  "Raw/interactions/interactions_dedup.json.gz"
}

for path in [
    "s3://s3-goodreads-data/Bronze",
    "s3://s3-goodreads-data/Silver",
    "s3://s3-goodreads-data/Gold",
    "s3://s3-goodreads-data/_metadata"
]:
    dbutils.fs.rm(path, recurse=True)

def worker_download(item):
    filename, s3_key = item
    url = base_url + filename

    try:
        s3_client.head_object(Bucket=BUCKET_NAME, Key=s3_key)
        print(f"[SKIPPED] {filename} — already in S3")
        return
    except ClientError as e:
        if e.response['Error']['Code'] != '404':
            print(f"[ERROR] {filename} — unexpected S3 error: {e}")
            return

    for attempt in range(1, 5):
        try:
            with requests.get(url, stream=True, timeout=(10, 300)) as r:
                r.raise_for_status()
                r.raw.decode_content = True
                s3_client.upload_fileobj(r.raw, BUCKET_NAME, s3_key, Config=TRANSFER_CONFIG)
            print(f"✅ FINISHED: {filename}")
            return
        except Exception as e:
            print(f"⚠️ Attempt {attempt}/5 failed for {filename}: {e}")
            time.sleep(2 ** attempt)
    print(f"❌ FAILED: {filename} after 4 attempts.")

# 3. EXECUTION
with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(worker_download, files_to_download.items())

# 4. PARQUET CONVERSION - for the intereactions 10 gb table
interactions_schema = StructType([
    StructField("book_id",    StringType(), True),
    StructField("date_added", StringType(), True),
    StructField("review_id",  StringType(), True),
    StructField("user_id",    StringType(), True),
])

try:
    spark.read.parquet(RAW_PARQUET).take(1)
    print("✅ Parquet already exists — skipping conversion.")
except:
    print("Converting interactions .json.gz → Parquet")
    spark.read \
        .schema(interactions_schema) \
        .json("s3://s3-goodreads-data/Raw/interactions/interactions_dedup.json.gz") \
        .repartition(8) \
        .write.format("parquet") \
        .mode("overwrite") \
        .save(RAW_PARQUET)
    print("✅ Parquet conversion done")